# Appendix: an example barley seed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path
import barley as B

OUT = Path("figures/4-barley"); OUT.mkdir(parents=True, exist_ok=True)
t = B.load_table(with_files=True)
i = int(np.random.default_rng(0).integers(len(t["files"])))   # one seed, chosen at random
img = B.load_seed(t["files"][i])
center, rot = B.paper_frame(np.argwhere(img).astype(float))
verts, faces = B.mesh(img, center)
v = verts @ rot * 0.127            # hand-built frame in mm (127 um voxels)
print(f"seed {i}: type {t['accession'][i]}, spike {t['panicle'][i]}, "
      f"{len(verts):,} vertices, {len(faces):,} triangles, extent (mm) {np.ptp(v, 0).round(2)}")

ls = LightSource(azdeg=315, altdeg=45)
tri = v[faces]
normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
normals /= np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12
VIEWS = [("oblique", 25, -60), ("top (length, width)", 90, -90),
         ("side (length, crease axis)", 0, -90), ("end (width, crease axis)", 0, 0)]
fig = plt.figure(figsize=(7.0, 1.9))
r = np.abs(v).max()
for k, (title, elev, azim) in enumerate(VIEWS):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d", proj_type="ortho")
    ax.view_init(elev=elev, azim=azim)
    light = np.array([np.cos(np.radians(elev + 30)) * np.cos(np.radians(azim + 30)),
                      np.cos(np.radians(elev + 30)) * np.sin(np.radians(azim + 30)),
                      np.sin(np.radians(elev + 30))])
    shade = 0.35 + 0.65 * np.clip(np.abs(normals @ light), 0, 1)
    colors = np.array([0.80, 0.62, 0.32])[None] * shade[:, None]
    pc = Poly3DCollection(tri, facecolors=np.clip(colors, 0, 1), edgecolors="none", linewidths=0)
    pc.set_rasterized(True)
    ax.add_collection3d(pc)
    lo, hi = v.min(0), v.max(0)
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
    ax.set_box_aspect(hi - lo, zoom=1.0)
    ax.set_axis_off()
    ax.set_title(title, fontsize=7.5, y=0.98)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0)
fig.savefig(OUT / "barley-seed.pdf", bbox_inches="tight", dpi=300)
